# Weather Report: Climate Change and Cyclone Ditwah in Sri Lanka

**Research Question:** Is Sri Lanka's climate changing, and was Cyclone Ditwah (26-30 Nov 2025) a symptom of that trend or a statistical outlier?

### Data Sources & Architecture
Data is sourced from the Open-Meteo Historical Weather API. It consists of ~31,000 days of reanalysis grid data (1940-2025) across 10 major locations in Sri Lanka.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Fix ModuleNotFoundError: Add the src/ folder to Python's search path
src_path = str(Path.cwd().parent / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import our custom project modules
from ditwah_weather.io import load_all
from ditwah_weather.config import LOCATIONS

# Load all 10 cities into one dataset
data_dir = Path.cwd().parent / "data"
df = load_all(data_dir)

print(f"Successfully loaded {len(df)} rows across {df['city'].nunique()} grid points.")
df.head(3)

Successfully loaded 314120 rows across 10 grid points.


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,date,location,city
0,1940-01-01,29.6,22.7,NaN,18.8,1940-01-01,Negombo,Negombo
1,1940-01-02,30.7,23.1,0.0,18.8,1940-01-02,Negombo,Negombo
2,1940-01-03,30.4,22.5,0.0,18.0,1940-01-03,Negombo,Negombo


## Act 1: The Warming Trend
Before looking at extreme rainfall events, we must establish if the background climate in Sri Lanka is changing. We calculate the annual maximum temperature anomalies (comparing each year to a standard 1961-1990 baseline) for all 10 grid points.

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from ditwah_weather.analysis import add_anomalies, annual_anomaly, decadal_trend

# 1. Calculate daily anomalies for all data
df_anom = add_anomalies(df, base=(1961, 1990), var='temperature_2m_max')

# 2. Calculate the warming trend per decade for all 10 cities
trends = []
city_annual_series = {}

for city in df_anom['city'].unique():
    # Filter the daily data for just one city
    city_df = df_anom[df_anom['city'] == city]
    
    # Calculate the annual average anomaly for this specific city
    # annual_anomaly() returns a pandas Series indexed by 'year'
    city_annual = annual_anomaly(city_df)
    city_annual_series[city] = city_annual
    
    # Calculate the trend
    slope, p_val = decadal_trend(city_annual.index, city_annual.values)
    trends.append({'City': city, 'Trend (°C / decade)': round(slope, 3), 'p-value': round(p_val, 4)})

# Display the results table
trend_df = pd.DataFrame(trends).sort_values('Trend (°C / decade)', ascending=False)
display(trend_df)

# 3. Visualisation: Anomaly Time Series for Dankotuwa
dankotuwa_annual = city_annual_series['Dankotuwa'].reset_index()
dankotuwa_annual['Rolling_10yr'] = dankotuwa_annual['anomaly'].rolling(10, center=True).mean()

fig = px.bar(dankotuwa_annual, x='year', y='anomaly', 
             title='Annual Temperature Anomaly in Dankotuwa (Baseline: 1961-1990)',
             labels={'year': 'Year', 'anomaly': 'Temperature Anomaly (°C)'},
             color='anomaly', color_continuous_scale='RdBu_r')

# Add the 10-year rolling mean as a trendline
fig.add_trace(go.Scatter(x=dankotuwa_annual['year'], y=dankotuwa_annual['Rolling_10yr'],
                         mode='lines', name='10-Year Rolling Mean',
                         line=dict(color='black', width=3)))

fig.update_layout(template='plotly_white', coloraxis_showscale=False)
fig.show()

,City,Trend (°C / decade),p-value
7,Trincomalee,0.152,0.0000
4,Nuwara_eliya,0.106,0.0065
5,Anuradhapura,0.095,0.0010
3,Ratnapura,0.094,0.0000
8,Galle,0.087,0.0000
9,Dankotuwa,0.061,0.0002
0,Negombo,0.058,0.0004
6,Batticaloa,0.056,0.0030
2,Kandy,0.051,0.0843
1,Colombo,0.043,0.0073
